# Axis 4 - Skills and AI: exploration

**Question:** which technologies and AI practices are linked to higher salaries in France?

This notebook prepares the page `views/axis_4_skills.py`: which questions have enough answers, which results are reliable, and which traps to avoid.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.cleaning import clean_survey
from src.population import paid_population, focus_country, comparison_group
from src.labels import TECH_FAMILIES, AI_TOOL_FAMILIES
from src import analysis

everyone = clean_survey(2026)
paid = paid_population(everyone)
france_paid = focus_country(paid)
france_all = focus_country(everyone)
eu_paid = comparison_group(paid, "European Union (without France)")

## 1. Enough answers?

The questions on technologies are answered by fewer people. For each family: how many French salaries, and how many technologies have at least 30 salaries (our reliability rule)?

In [2]:
rows = []
for family, column in {**TECH_FAMILIES, **AI_TOOL_FAMILIES}.items():
    table = analysis.median_by_answer(france_paid, column)
    rows.append({"family": family, "salaries in France": france_paid[column].notna().sum(),
                 "technologies": len(table), "with 30+ salaries": table["reliable"].sum()})
pd.DataFrame(rows)

,family,salaries in France,technologies,with 30+ salaries
0,Languages,329,47,14
1,Databases,296,29,8
2,Cloud and infrastructure,255,57,26
3,Web frameworks,250,36,11
4,Code editors,333,30,14
5,AI models (LLM vendors),280,17,6
6,Coding agents and assistants,286,26,6


**Decision:** we only show the technologies with at least 30 salaries, and we say how many are hidden.

## 2. Salary of the users of each language in France

In [3]:
answered = france_paid[france_paid["LanguagesHaveWorkedWith"].notna()]
by_language = analysis.median_by_answer(answered, "LanguagesHaveWorkedWith")
print("Median of all the people who answered:", analysis.salary_summary(answered)["median"])
by_language[by_language["reliable"]].sort_values("median", ascending=False)

Median of all the people who answered: 55000.0


,answer,n,median,q1,q3,reliable
39,Rust,65,66000.0,50000.0,85000.0,True
16,Go,49,65000.0,50000.0,80000.0,True
5,C++,55,61200.0,41500.0,77500.0,True
43,TypeScript,162,56780.0,42000.0,71500.0,True
2,Bash/Shell (all shells),185,55000.0,42000.0,75000.0,True
36,Python,181,54000.0,40000.0,75000.0,True
34,PowerShell,76,53500.0,41750.0,72750.0,True
40,SQL,202,52640.0,42000.0,70000.0,True
18,HTML/CSS,203,52000.0,42000.0,70000.0,True
21,JavaScript,210,51000.0,41000.0,70000.0,True


Rust and Go users earn the most. **Trap:** maybe they are just more experienced? We compare the median experience of the users of each language.

In [4]:
exploded = answered.assign(answer=answered["LanguagesHaveWorkedWith"].str.split(";")).explode("answer")
experience = exploded.groupby("answer")["WorkExp"].median().rename("median experience (years)")
reliable = by_language[by_language["reliable"]].set_index("answer")
reliable[["n", "median"]].join(experience).sort_values("median", ascending=False)

,n,median,median experience (years)
answer,,,
Rust,65,66000.0,11.0
Go,49,65000.0,11.0
C++,55,61200.0,11.0
TypeScript,162,56780.0,11.0
Bash/Shell (all shells),185,55000.0,12.0
Python,181,54000.0,11.0
PowerShell,76,53500.0,12.0
SQL,202,52640.0,12.0
HTML/CSS,203,52000.0,11.0


All the languages have about the same median experience (11-12 years): the higher salaries of Rust and Go users are **not** explained by experience. It is still a correlation, not a cause (sector, company, role can play).

## 3. Do AI users earn more?

In [5]:
analysis.median_by_group(france_paid, "UsesAI")

,UsesAI,n,median,q1,q3,reliable
0,False,86,45000.0,38000.0,60000.0,True
1,True,327,58000.0,43000.0,80000.0,True


Same trap: is it experience? We compare at equal experience, in France and in the rest of the EU (more respondents).

In [6]:
for name, df in [("France", france_paid), ("Rest of the EU", eu_paid)]:
    table = (df.dropna(subset=["UsesAI"])
               .groupby(["ExperienceLevel", "UsesAI"], observed=True)["SalaryEUR"]
               .agg(["median", "size"]).unstack())
    print(name)
    display(table)

France


median           size      
UsesAI             False    True  False True 
ExperienceLevel                              
0-2 years        30000.0  32000.0     7    13
3-5 years        36000.0  40000.0    13    40
6-10 years       45000.0  54000.0    23    85
11-20 years      50000.0  62000.0    30   117
20+ years        66000.0  75000.0    13    69

Rest of the EU


median           size      
UsesAI             False    True  False True 
ExperienceLevel                              
0-2 years        36159.0  40000.0    25   121
3-5 years        36159.0  50000.0    39   356
6-10 years       56000.0  63139.0    77   750
11-20 years      68899.0  75000.0   106  1152
20+ years        65000.0  80000.0    75   657

At every experience level, AI users earn more, in France and in the rest of the EU. In France, the groups of non-users are small (7 to 30 people): the page shows them with empty markers, and the rest of the EU confirms the result with bigger groups.

## 4. Opinions and tools

In [7]:
analysis.compare_shares({"France": france_all, "Rest of the EU": comparison_group(everyone, "European Union (without France)")},
                        "AISentiment", multiselect=False)

,group,answer,count,percent,n,reliable
0,France,Very unfavorable,33,6.860707,481,True
1,France,Somewhat unfavorable,96,19.958420,481,True
2,France,Neutral,83,17.255717,481,True
3,France,Somewhat favorable,134,27.858628,481,True
4,France,Very favorable,135,28.066528,481,True
5,Rest of the EU,Very unfavorable,318,6.809422,4670,True
6,Rest of the EU,Somewhat unfavorable,779,16.680942,4670,True
7,Rest of the EU,Neutral,692,14.817987,4670,True
8,Rest of the EU,Somewhat favorable,1492,31.948608,4670,True
9,Rest of the EU,Very favorable,1389,29.743041,4670,True


In [8]:
analysis.count_multiselect(france_all["LLMHaveWorkedWith"]).head(8)

,answer,count,percent
0,Anthropic,321,76.610979
1,OpenAI,303,72.315036
2,Google AI,167,39.856802
3,Mistral AI,137,32.696897
4,Microsoft,82,19.570406
5,DeepSeek,76,18.138425
6,Alibaba Qwen,33,7.875895
7,xAI,20,4.773270


Mistral AI, a French company, is used by about a third of the French respondents who answered.

## 5. Languages people want to learn

In [9]:
used = analysis.count_multiselect(france_all["LanguagesHaveWorkedWith"]).set_index("answer")["percent"]
wanted = analysis.count_multiselect(france_all["LanguagesWantToWorkWith"]).set_index("answer")["percent"]
gap = pd.DataFrame({"used": used, "wanted": wanted}).dropna()
gap["wanted - used"] = gap["wanted"] - gap["used"]
gap.sort_values("wanted - used", ascending=False).head(8).round(1)

,used,wanted,wanted - used
answer,,,
Rust,22.5,40.7,18.2
Zig,1.7,9.1,7.4
Go,15.5,20.1,4.6
Elixir,2.3,5.9,3.5
Gleam,1.6,4.4,2.9
COBOL,0.6,2.3,1.8
Dart,3.7,5.2,1.5
Assembly,9.9,11.2,1.3


Rust is the language French developers most want to learn (41% want it, 23% use it): a signal for training offers.

## 6. Languages 2025 → 2026 (harmonised)

Only the languages offered both years are compared (7 languages are new in 2026).

In [10]:
france_2025 = focus_country(clean_survey(2025))
trend = analysis.evolution_of_shares(france_2025["LanguagesHaveWorkedWith"], france_all["LanguagesHaveWorkedWith"], 2025, 2026)
trend.sort_values("percent_2026", ascending=False).head(12).round(1)

,answer,n_2025,percent_2025,n_2026,percent_2026,change_points
19,JavaScript,1307,63.4,515,62.5,-0.8
30,Python,1307,62.7,515,61.0,-1.8
17,HTML/CSS,1307,62.3,515,60.2,-2.1
34,SQL,1307,57.0,515,57.5,0.5
2,Bash/Shell (all shells),1307,56.2,515,57.1,0.9
37,TypeScript,1307,44.6,515,44.1,-0.5
18,Java,1307,31.4,515,29.1,-2.3
3,C,1307,26.9,515,24.7,-2.3
26,PHP,1307,22.3,515,24.3,2.0
5,C++,1307,26.2,515,23.9,-2.4
